In this notebook we will go through the reproduction of the study mentioned in the paper. 
First thing we will do is get the datasets and the needed dependencies. 

In [47]:
%pip install aif360 pandas numpy scikit-learn matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


In [48]:
from aif360.sklearn.datasets import (
    fetch_adult,
    fetch_german,
    fetch_compas)

adult = fetch_adult()
german = fetch_german()
compas = fetch_compas()

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
import random
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score
from sklearn.dummy import DummyClassifier

SEED = 69

random.seed(SEED)
np.random.seed(SEED)


Now that we have the datasets we can check what the privileged and unprivileged groups are within the datasets on the favorable outcomes used in the study. 
First we need to be able to get the attributes.

In [49]:
def get_protected_attribute(dataset, attribute):
    if attribute in dataset.X.columns:
        values = dataset.X[attribute]
    else:
        values = dataset.X.index.get_level_values(attribute)
    values = pd.Series(values).reset_index(drop=True)
    return values

Now we can check the favourable outcomes per group in the attributes and make a decision. 

In [50]:
def calculate_group_rates(dataset, attribute, favorable_label):
    groups = get_protected_attribute(dataset, attribute)
    favorable = (dataset.y == favorable_label).astype(int)
    favorable = favorable.reset_index(drop=True)
    data = pd.DataFrame({
        "Group": groups,
        "Favorable": favorable})
    rates = data.groupby("Group")["Favorable"].mean()
    return rates

In [51]:
print("ADULT - SEX")
adult_sex_rates = calculate_group_rates(
    adult, "sex", ">50K")
print((adult_sex_rates * 100).round(2))
print("\nADULT - RACE")
adult_race_rates = calculate_group_rates(
    adult, "race", ">50K")
print((adult_race_rates * 100).round(2))

ADULT - SEX
Group
Female    11.36
Male      31.25
Name: Favorable, dtype: float64

ADULT - RACE
Group
Amer-Indian-Eskimo    12.18
Asian-Pac-Islander    28.32
Black                 12.63
Other                 12.75
White                 26.24
Name: Favorable, dtype: float64


For the Adult set we see that the unprivileged groups are "female" for sex and anything other than "white" and "Asian-Pac-Islander"  

In [52]:
print("GERMAN CREDIT - SEX")
german_sex_rates = calculate_group_rates(
    german, "sex", "good")
print((german_sex_rates * 100).round(2))
print("\nGERMAN CREDIT - AGE")
german_age_rates = calculate_group_rates(
    german, "age", "good")
print((german_age_rates * 100).round(2))

GERMAN CREDIT - SEX
Group
female    64.84
male      72.32
Name: Favorable, dtype: float64

GERMAN CREDIT - AGE
Group
19     50.00
20     64.29
21     64.29
22     59.26
23     58.33
24     56.82
25     53.66
26     72.00
27     74.51
28     65.12
29     59.46
30     72.50
31     71.05
32     73.53
33     60.61
34     65.62
35     85.00
36     84.62
37     72.41
38     83.33
39     71.43
40     76.00
41     76.47
42     63.64
43     70.59
44     70.59
45     80.00
46     77.78
47     70.59
48     75.00
49     92.86
50     75.00
51     87.50
52     88.89
53     28.57
54     80.00
55     62.50
56    100.00
57     66.67
58     60.00
59     66.67
60     50.00
61     57.14
62    100.00
63     87.50
64    100.00
65     80.00
66     60.00
67    100.00
68     33.33
70    100.00
74     75.00
75    100.00
Name: Favorable, dtype: float64


For the German Credit set we see that the unprivileged groups are "female" for sex and we will take a cut off of 25 years old for the age

In [53]:
print("COMPAS - SEX")
compas_sex_rates = calculate_group_rates(
    compas, "sex", "Survived")
print((compas_sex_rates * 100).round(2))
print("\nCOMPAS - RACE")
compas_race_rates = calculate_group_rates(
    compas, "race", "Survived")
print((compas_race_rates * 100).round(2))

COMPAS - SEX
Group
Male      52.02
Female    64.79
Name: Favorable, dtype: float64

COMPAS - RACE
Group
African-American    47.65
Asian               74.19
Caucasian           60.86
Hispanic            62.87
Native American     54.55
Other               63.85
Name: Favorable, dtype: float64


For the COMPAS set we see that the unprivileged groups are "female" for sex and "african-american" for race. 

Now that we have our groups we can make 6 different experimental groups to calculate the fairness of the metrics in the original datasets.

In [54]:
experimentgroups = [
    {
        "name": "Adult - Sex",
        "dataset": adult,
        "attribute": "sex",
        "unprivileged": ["Female"],
        "privileged": ["Male"],
        "favorable": ">50K"},
    {
        "name": "Adult - Race",
        "dataset": adult,
        "attribute": "race",
        "unprivileged": ["Amer-Indian-Eskimo", "Black", "Other"],
        "privileged": ["White", "Asian-Pac-Islander"],
        "favorable": ">50K"},
    {
        "name": "German Credit - Sex",
        "dataset": german,
        "attribute": "sex",
        "unprivileged": ["female"],
        "privileged": ["male"],
        "favorable": "good"},
    {
        "name": "German Credit - Age",
        "dataset": german,
        "attribute": "age",
        "unprivileged": ["25 or younger"],
        "privileged": ["Older than 25"],
        "favorable": "good"},
    {
        "name": "COMPAS - Sex",
        "dataset": compas,
        "attribute": "sex",
        "unprivileged": ["Female"],
        "privileged": ["Male"],
        "favorable": "Survived"},
    {
        "name": "COMPAS - Race",
        "dataset": compas,
        "attribute": "race",
        "unprivileged": ["African-American"],
        "privileged": ["Caucasian", "Hispanic", "Other", "Asian", "Native American"],
        "favorable": "Survived"
    }
]

Now we will make a funtion that checks which people are in which group

In [55]:
def get_protected_attribute(dataset, attribute):
    if attribute in dataset.X.columns:
        values = dataset.X[attribute]
    else:
        values = dataset.X.index.get_level_values(attribute)
    return pd.Series(values).reset_index(drop=True)

And a function that assigns them to their group

In [56]:
def get_group_masks(dataset, experiment):
    attribute = experiment["attribute"]
    values = get_protected_attribute(dataset, attribute)
    if attribute == "age":
        unprivileged = values <= 25
        privileged = values > 25
    elif experiment["name"] == "Adult - Race":
        unprivileged = values.isin(
            experiment["unprivileged"])
        privileged = values.isin(
            experiment["privileged"])
    elif experiment["name"] == "COMPAS - Race":
        unprivileged = values.isin(
            experiment["unprivileged"])
        privileged = ~unprivileged
    else:
        unprivileged = values.isin(
            experiment["unprivileged"])
        privileged = values.isin(
            experiment["privileged"])
    return unprivileged, privileged

Now that we have all records in their respective groups it is time to calculate the bias within the data. 
We will start by making functions for the demographic parity difference (DP_Diff) and the disparate impact ratio (DIR).

In [57]:
def calculate_fairness(y, unprivileged, privileged):

    # rate for unprivileged group
    unprivileged_rate = y[unprivileged].mean()

    # rate for privileged group
    privileged_rate = y[privileged].mean()

    # demographic parity difference
    dp_diff = unprivileged_rate - privileged_rate

    # disparate impact ratio
    if privileged_rate == 0:
        dir_value = np.nan
    else:
        dir_value = unprivileged_rate / privileged_rate

    return dp_diff, dir_value

We will also make a function for the consistency
For this we will use 5 nearest neighbours

In [58]:
def calculate_consistency(X, y):
    X_encoded = pd.get_dummies(X)
    neighbors = NearestNeighbors(n_neighbors=6)
    neighbors.fit(X_encoded)
    neighbor_indices = neighbors.kneighbors(return_distance=False)
    neighbor_indices = neighbor_indices[:, 1:]

    y = np.array(y)
    neighbor_labels = y[neighbor_indices]
    differences = np.abs(y[:, None] - neighbor_labels)
    consistency = 1 - differences.mean()

    return consistency

Now we will run these functions over the datasets to get the fairness. 

In [59]:
results_before = []

for experiment in experimentgroups:
    name = experiment["name"]
    dataset = experiment["dataset"]
    y = (dataset.y == experiment["favorable"]).astype(int)
    y = y.reset_index(drop=True)

    unprivileged, privileged = get_group_masks(
        dataset, experiment)
    dp_diff, dir_value = calculate_fairness(
        y, unprivileged, privileged)
    X = dataset.X.reset_index(drop=True).copy()
    for column in dataset.X.index.names:
        if column is not None and column not in X.columns:
            X[column] = dataset.X.index.get_level_values(column).to_numpy()

    consistency = calculate_consistency(X, y)
    results_before.append({
        "Dataset": name,
        "DP_Diff": dp_diff,
        "DIR": dir_value,
        "Consistency": consistency})
results_before_df = pd.DataFrame(results_before)
results_before_df.round(4)

,Dataset,DP_Diff,DIR,Consistency
0,Adult - Sex,-0.1989,0.3635,0.8096
1,Adult - Race,-0.1370,0.4790,0.8096
2,German Credit - Sex,-0.0748,0.8966,0.5960
3,German Credit - Age,-0.1494,0.7948,0.5960
4,COMPAS - Sex,0.1277,1.2454,0.5903
5,COMPAS - Race,-0.1400,0.7729,0.5903


Now that we have the fairness metrics of the original datasets it is time to perform a train/test split with stratification.
We will first make the function to do so. 

In [60]:
def split_dataset(dataset, random_state=SEED):
    X = dataset.X.reset_index(drop=True)
    y = dataset.y.reset_index(drop=True)

    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=random_state,
        stratify=y)

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val,
        y_train_val,
        test_size=0.25,
        random_state=random_state,
        stratify=y_train_val)

    return (
        X_train.reset_index(drop=True),
        X_val.reset_index(drop=True),
        X_test.reset_index(drop=True),
        y_train.reset_index(drop=True),
        y_val.reset_index(drop=True),
        y_test.reset_index(drop=True)
    )

Now we will run it over our datasets. 

In [61]:
adult_split = split_dataset(adult, random_state=SEED)
german_split = split_dataset(german, random_state=SEED)
compas_split = split_dataset(compas, random_state=SEED)

In [62]:
for name, split in [
    ("Adult", adult_split),
    ("German Credit", german_split),
    ("COMPAS", compas_split)
]:
    X_train, X_val, X_test, y_train, y_val, y_test = split

    print(f"\n{name}")
    print(f"Training:   {len(X_train)} ({len(X_train) / (len(X_train) + len(X_val) + len(X_test)):.0%})")
    print(f"Validation: {len(X_val)}")
    print(f"Testing:    {len(X_test)}")


Adult
Training:   27132 (60%)
Validation: 9045
Testing:    9045

German Credit
Training:   600 (60%)
Validation: 200
Testing:    200

COMPAS
Training:   3699 (60%)
Validation: 1234
Testing:    1234


In [63]:
def encode_protected_attribute(X, task_name):
    X = X.copy()
    if task_name == "Adult - Sex":
        protected = (X["sex"] == "Male").astype(int)
        X = X.drop(columns=["sex"])
    elif task_name == "Adult - Race":
        protected = X["race"].isin(
            ["White", "Asian-Pac-Islander"]
        ).astype(int)
        X = X.drop(columns=["race"])
    elif task_name == "German Credit - Sex":
        protected = (X["sex"] == "male").astype(int)
        X = X.drop(columns=["sex"])
    elif task_name == "German Credit - Age":
        protected = (X["age"] > 25).astype(int)
        X = X.drop(columns=["age"])
    elif task_name == "COMPAS - Sex":
        protected = (X["sex"] == "Male").astype(int)
        X = X.drop(columns=["sex"])
    elif task_name == "COMPAS - Race":
        protected = (X["race"] != "African-American").astype(int)
        X = X.drop(columns=["race"])
    else:
        raise ValueError("Unknown experiment: " + task_name)
    X["protected_group"] = protected
    return X

We will now split the tasks

In [64]:
experiment_splits = {
    "Adult - Sex": adult_split,
    "Adult - Race": adult_split,
    "German Credit - Sex": german_split,
    "German Credit - Age": german_split,
    "COMPAS - Sex": compas_split,
    "COMPAS - Race": compas_split}
favorable_labels = {
    experiment["name"]: experiment["favorable"]
    for experiment in experimentgroups}
task_splits = {}
for task_name, split in experiment_splits.items():
    X_train, X_val, X_test, y_train, y_val, y_test = split
    favorable_label = favorable_labels[task_name]
    y_train = (y_train == favorable_label).astype(int)
    y_val = (y_val == favorable_label).astype(int)
    y_test = (y_test == favorable_label).astype(int)

    task_splits[task_name] = (
        encode_protected_attribute(X_train, task_name),
        encode_protected_attribute(X_val, task_name),
        encode_protected_attribute(X_test, task_name),
        y_train,
        y_val,
        y_test)

In [65]:
print("Experiments prepared:", list(task_splits.keys()))

Experiments prepared: ['Adult - Sex', 'Adult - Race', 'German Credit - Sex', 'German Credit - Age', 'COMPAS - Sex', 'COMPAS - Race']


Because our study is about a novel post-processing method we need to process the data:
We will use only a logistic regression in this pipeline to keep it short.

For each protected attribute, we create a binary protected_group feature:
0 = unprivileged and 1 = privileged.

We also convert the target into binary values:
0 = unfavorable and 1 = favorable.

In [66]:
def train_baseline_model(X_train, y_train):
    numerical_columns = X_train.select_dtypes(
        include=["int64", "float64"]).columns
    categorical_columns = X_train.select_dtypes(include=["object", "category", "bool"]).columns
    numerical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))])
    preprocessor = ColumnTransformer([
        ("num", numerical_pipeline, numerical_columns),
        ("cat", categorical_pipeline, categorical_columns)])
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, random_state=SEED))])
    model.fit(X_train, y_train)
    return model

Now we will train one logistic regression model for each of the six experiments.

In [67]:
task_models = {}

for task_name, split in task_splits.items():
    X_train, X_val, X_test, y_train, y_val, y_test = split
    task_models[task_name] = train_baseline_model(
        X_train,
        y_train)

Now we will generate predictions on the validation data for each experiment.

In [68]:
validation_predictions = {}
for task_name, split in task_splits.items():
    X_train, X_val, X_test, y_train, y_val, y_test = split
    validation_predictions[task_name] = task_models[task_name].predict(X_val)

We now implement the IGD post-processing method for all six experiments.
First we calculate the individual bias scores

In [69]:
def calculate_individual_bias(model, X):
    X_counterfactual = X.copy()
    X_counterfactual["protected_group"] = 1
    original_scores = model.predict_proba(X)[:, 1]
    counterfactual_scores = model.predict_proba(X_counterfactual)[:, 1]
    bias_scores = counterfactual_scores - original_scores
    return bias_scores


Now we select the bias threshold using the validation data. These are the same as in the study itself

In [74]:

def select_igd_threshold(
    model,
    X_val,
    bias_scores,
    unprivileged,
    privileged,
    target_lower=0.8,
    target_upper=1.25):
    validation_predictions = model.predict(X_val)
    X_val_unprivileged = X_val.loc[unprivileged].copy()
    X_val_counterfactual = X_val_unprivileged.copy()
    X_val_counterfactual["protected_group"] = 1
    counterfactual_predictions = model.predict(X_val_counterfactual)
    thresholds = np.concatenate(
        ([-np.inf], np.sort(np.unique(bias_scores)), [np.inf]))
    threshold_results = []
    for threshold in thresholds:
        adjusted_predictions = validation_predictions.copy()
        biased_samples = bias_scores > threshold
        unprivileged_indices = np.where(unprivileged)[0]
        adjusted_predictions[
            unprivileged_indices[biased_samples]
        ] = counterfactual_predictions[biased_samples]
        _, dir_value = calculate_fairness(
            adjusted_predictions,
            unprivileged,
            privileged)
        if np.isnan(dir_value):
            distance = np.inf
        elif dir_value < target_lower:
            distance = target_lower - dir_value
        elif dir_value > target_upper:
            distance = dir_value - target_upper
        else:
            distance = 0
        threshold_results.append({
            "threshold": threshold,
            "DIR": dir_value,
            "distance": distance})

    threshold_results = pd.DataFrame(threshold_results)
    threshold_results["distance_from_1"] = (
        threshold_results["DIR"] - 1
    ).abs()
    best_result = threshold_results.sort_values(
        ["distance", "distance_from_1"],
        ascending=[True, True]
    ).iloc[0]
    selected_threshold = best_result["threshold"]
    return selected_threshold, best_result["DIR"], threshold_results


Now we train the bias detector


In [71]:
def train_igd_bias_detector(X_val, bias_scores, bias_threshold, unprivileged):
    # Keep only unprivileged validation samples
    X_bias_train = X_val.loc[unprivileged].drop(
        columns=["protected_group"]
    ).copy()
    y_bias_train = (bias_scores > bias_threshold).astype(int)
    if len(np.unique(y_bias_train)) == 1:
        bias_detector = DummyClassifier(
            strategy="constant",
            constant=int(y_bias_train[0]))
        bias_detector.fit(X_bias_train, y_bias_train)
    else:
        bias_detector = train_baseline_model(X_bias_train, y_bias_train)
    return bias_detector, X_bias_train, y_bias_train


Now that we have trained our bias detector we can use it to apply it to the testdata itself

In [72]:
def apply_igd(model, bias_detector, X_test):
    original_predictions = model.predict(X_test)
    test_unprivileged = X_test["protected_group"] == 0
    test_privileged = X_test["protected_group"] == 1
    igd_predictions = original_predictions.copy()
    X_test_unprivileged = X_test.loc[test_unprivileged].drop(
        columns=["protected_group"]
    ).copy()
    test_bias_predictions = bias_detector.predict(
        X_test_unprivileged)
    X_test_counterfactual = X_test.loc[test_unprivileged].copy()
    X_test_counterfactual["protected_group"] = 1
    counterfactual_predictions = model.predict(
        X_test_counterfactual)
    unprivileged_indices = np.where(test_unprivileged)[0]
    samples_to_correct = test_bias_predictions == 1
    igd_predictions[
        unprivileged_indices[samples_to_correct]
    ] = counterfactual_predictions[samples_to_correct]

    return( igd_predictions,
        original_predictions,
        test_bias_predictions,
        samples_to_correct)


Now that we have trained the IGD we run it on all 6 our experiment groups 

In [ ]:
igd_models = {}
igd_thresholds = {}
igd_validation_dirs = {}
igd_test_predictions = {}
igd_details = {}
for task_name, split in task_splits.items():
    X_train, X_val, X_test, y_train, y_val, y_test = split
    model = task_models[task_name]
    unprivileged = X_val["protected_group"] == 0
    privileged = X_val["protected_group"] == 1
    X_val_unprivileged = X_val.loc[unprivileged].copy()
    bias_scores = calculate_individual_bias(
        model,
        X_val_unprivileged)
    bias_threshold, validation_dir, threshold_results = (
        select_igd_threshold(
            model,
            X_val,
            bias_scores,
            unprivileged,
            privileged))
    bias_detector, X_bias_train, y_bias_train = (
        train_igd_bias_detector(
            X_val,
            bias_scores,
            bias_threshold,
            unprivileged))

    (igd_predictions,
        original_predictions,
        test_bias_predictions,
        samples_to_correct) = apply_igd(
        model,
        bias_detector,
        X_test)

    igd_models[task_name] = bias_detector
    igd_thresholds[task_name] = bias_threshold
    igd_validation_dirs[task_name] = validation_dir
    igd_test_predictions[task_name] = igd_predictions
    igd_details[task_name] = {
        "threshold_results": threshold_results,
        "bias_training_labels": y_bias_train,
        "test_bias_predictions": test_bias_predictions,
        "samples_to_correct": samples_to_correct,
        "original_predictions": original_predictions}



Adult - Sex
Selected bias threshold: -inf
Validation DIR after simulated correction: 0.47316816272860224
Bias detector training samples: 2912
Biased validation samples: 2912
Test samples corrected: 3031

Adult - Race
Selected bias threshold: -inf
Validation DIR after simulated correction: 0.40171896872461305
Bias detector training samples: 1055
Biased validation samples: 1055
Test samples corrected: 1004

German Credit - Sex
Selected bias threshold: -inf
Validation DIR after simulated correction: 0.9630893300248139
Bias detector training samples: 62
Biased validation samples: 62
Test samples corrected: 67

German Credit - Age
Selected bias threshold: -inf
Validation DIR after simulated correction: 0.9575892857142857
Bias detector training samples: 35
Biased validation samples: 35
Test samples corrected: 42

COMPAS - Sex
Selected bias threshold: -inf
Validation DIR after simulated correction: 1.3784897025171625
Bias detector training samples: 230
Biased validation samples: 230
Test sam

Now that we have run the full IGD over our data we can check how it performed by crossvalidatiing it on the fairness metrics calculated before logistic regression. 


In [ ]:
igd_results = []
for task_name, split in task_splits.items():
    X_train, X_val, X_test, y_train, y_val, y_test = split

    predictions_before = task_models[task_name].predict(X_test)
    predictions_after = igd_test_predictions[task_name]
    test_unprivileged = X_test["protected_group"] == 0
    test_privileged = X_test["protected_group"] == 1
    dp_diff_before, dir_before = calculate_fairness(
        predictions_before,
        test_unprivileged,
        test_privileged)

    dp_diff_after, dir_after = calculate_fairness(
        predictions_after,
        test_unprivileged,
        test_privileged)
    consistency_before = calculate_consistency(
        X_test,
        predictions_before)
    consistency_after = calculate_consistency(
        X_test,
        predictions_after)
    balanced_accuracy_before = balanced_accuracy_score(
        y_test,
        predictions_before)
    balanced_accuracy_after = balanced_accuracy_score(
        y_test,
        predictions_after)

    igd_results.append({
        "Dataset": task_name,
        "DP_Diff Before": dp_diff_before,
        "DP_Diff After": dp_diff_after,
        "DIR Before": dir_before,
        "DIR After": dir_after,
        "Consistency Before": consistency_before,
        "Consistency After": consistency_after,
        "Balanced Accuracy Before": balanced_accuracy_before,
        "Balanced Accuracy After": balanced_accuracy_after})

igd_results_df = pd.DataFrame(igd_results)

igd_results_df.round(4)


,Dataset,DP_Diff Before,DP_Diff After,DIR Before,DIR After,Consistency Before,Consistency After,Balanced Accuracy Before,Balanced Accuracy After
0,Adult - Sex,-0.1901,-0.1554,0.3035,0.4304,0.8830,0.8761,0.7713,0.7775
1,Adult - Race,-0.1433,-0.1294,0.3630,0.4250,0.8833,0.8827,0.7708,0.7721
2,German Credit - Sex,-0.0131,0.0018,0.9827,1.0024,0.6240,0.6270,0.6464,0.6500
3,German Credit - Age,-0.2990,-0.1085,0.6366,0.8681,0.6270,0.6720,0.6381,0.5952
4,COMPAS - Sex,0.2461,0.1189,1.4481,1.2164,0.8303,0.8342,0.6607,0.6562
5,COMPAS - Race,-0.2044,-0.1855,0.7081,0.7352,0.8308,0.8308,0.6580,0.6572
